# TP — Modèles de langage basés sur les N-grammes

**Traitement Automatique du Langage Naturel (NLP)** — Master, ISI  
Année académique 2026–2027

---

## Objectif

Construire pas à pas un modèle de langage statistique à partir d'un corpus, puis
l'utiliser pour cinq tâches NLP : prédiction du mot suivant, génération de texte,
évaluation de phrases, comparaison de phrases et correction contextuelle.

Chaîne de traitement :

`Corpus → Tokenisation → N-grammes → Comptage → Probabilités → Modèle → Applications`

## Organisation du code

Les fonctions sont définies dans le module **`modele_langage.py`** et importées ici.
Le notebook sert à exécuter, observer et commenter ; il ne redéfinit rien.

**Aucune bibliothèque NLP n'est utilisée** : tout est construit avec les structures
de base de Python (dictionnaires, listes, `Counter`).

In [10]:
# Rechargement automatique du module : les modifications de modele_langage.py
# sont prises en compte sans redémarrer le kernel.
%load_ext autoreload
%autoreload 2

from modele_langage import *
from collections import Counter

CORPUS_PATH = "data/corpus.txt"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---

## Partie 1 — Prétraitement du corpus

Le corpus brut est une suite de phrases en français. Un programme ne sait pas
manipuler du texte : il faut le transformer en une **structure discrète**.

Quatre opérations :

1. **Mise en minuscules** — sans elle, `Le` et `le` seraient deux entrées
   distinctes du vocabulaire, et les comptages seraient dispersés.
2. **Suppression de la ponctuation** — le point de `poisson.` ferait de ce mot un
   token différent de `poisson`.
3. **Tokenisation** — découpage en unités d'analyse.
4. **Ajout des marqueurs `<s>` et `</s>`** — sans eux, le modèle ne saurait ni
   quels mots peuvent *commencer* une phrase, ni quand une phrase se *termine*.
   Ces deux marqueurs sont traités comme des tokens à part entière.

In [11]:
# --- Question 1 : afficher les tokens de chaque phrase ---
corpus = charger_corpus(CORPUS_PATH)

for i, phrase in enumerate(corpus, 1):
    print(f"Phrase {i} ({len(phrase)} tokens) : {phrase}")

Phrase 1 (7 tokens) : ['<s>', 'le', 'chat', 'mange', 'du', 'poisson', '</s>']
Phrase 2 (7 tokens) : ['<s>', 'le', 'chat', 'aime', 'le', 'poisson', '</s>']
Phrase 3 (8 tokens) : ['<s>', 'le', 'chien', 'mange', 'de', 'la', 'viande', '</s>']
Phrase 4 (7 tokens) : ['<s>', 'le', 'chien', 'aime', 'la', 'viande', '</s>']
Phrase 5 (8 tokens) : ['<s>', 'le', 'chat', 'joue', 'dans', 'le', 'jardin', '</s>']
Phrase 6 (8 tokens) : ['<s>', 'le', 'chien', 'joue', 'dans', 'le', 'jardin', '</s>']


In [12]:
# --- Questions 2 et 3 : vocabulaire et sa taille ---
vocabulaire = construire_vocabulaire(corpus)
V = len(vocabulaire)

print(f"Vocabulaire ({V} types) :\n")
for mot in vocabulaire:
    print(f"  {mot}")

print(f"\nTaille du vocabulaire : V = {V}")
print(f"  dont 2 marqueurs -> {V - 2} mots réels")

Vocabulaire (15 types) :

  </s>
  <s>
  aime
  chat
  chien
  dans
  de
  du
  jardin
  joue
  la
  le
  mange
  poisson
  viande

Taille du vocabulaire : V = 15
  dont 2 marqueurs -> 13 mots réels


In [13]:
# --- Questions 4 et 5 : nombre de tokens et fréquences ---
tokens = aplatir(corpus)
N = len(tokens)
frequences = Counter(tokens)

print(f"Nombre total de tokens (occurrences) : N = {N}")
print(f"  dont marqueurs : {2 * len(corpus)}  |  mots réels : {N - 2 * len(corpus)}")
print(f"\nRatio N/V = {N}/{V} = {N/V:.2f} occurrences par type\n")

print(f"{'token':10s} {'fréquence':>10s}")
print("-" * 21)
for mot, freq in frequences.most_common():
    print(f"{mot:10s} {freq:>10d}")

Nombre total de tokens (occurrences) : N = 45
  dont marqueurs : 12  |  mots réels : 33

Ratio N/V = 45/15 = 3.00 occurrences par type

token       fréquence
---------------------
le                  9
<s>                 6
</s>                6
chat                3
chien               3
mange               2
poisson             2
aime                2
la                  2
viande              2
joue                2
dans                2
jardin              2
du                  1
de                  1


### Réponses — Partie 1

**Q1. Tokens de chaque phrase.** Voir la sortie ci-dessus. Chaque phrase devient une
liste de tokens encadrée par `<s>` et `</s>`. Les phrases font 7 ou 8 tokens.

**Q2. Vocabulaire.** `</s>`, `<s>`, `aime`, `chat`, `chien`, `dans`, `de`, `du`,
`jardin`, `joue`, `la`, `le`, `mange`, `poisson`, `viande`.

**Q3. Taille du vocabulaire : V = 15** (13 mots réels + 2 marqueurs).

> **Choix de modélisation.** Les marqueurs sont inclus dans le vocabulaire. C'est
> cohérent : `</s>` est un token que le modèle doit pouvoir *prédire* (il faut bien
> qu'une phrase générée s'arrête). Ce choix a une conséquence directe en Partie 10,
> où V apparaît au dénominateur du lissage de Laplace.

**Q4. Nombre total de tokens : N = 45** — soit 33 mots réels et 12 marqueurs
(6 phrases × 2).

**Q5. Différence entre vocabulaire et nombre de tokens.**

Le vocabulaire compte les **types** (mots distincts) ; N compte les **occurrences**.
Le mot `le` est **une seule** entrée du vocabulaire mais apparaît **9 fois** dans le
corpus : il pèse 20 % des tokens à lui seul.

D'où N = 45 pour V = 15, soit 3 occurrences par type en moyenne.

Cette distinction est le fondement de tout ce qui suit. Un modèle N-gramme estime des
probabilités **en divisant des occurrences par des occurrences** : plus le ratio N/V
est élevé, plus chaque estimation repose sur des observations nombreuses, donc plus
elle est fiable. Ici le ratio est très faible (3) — c'est pourquoi ce corpus jouet
produira des probabilités extrêmes (0 ou 1) que l'on corrigera par lissage.

Quand le corpus grandit, N croît linéairement tandis que V sature : c'est la **loi de
Heaps**. On ne cesse jamais de rencontrer des mots nouveaux, mais de plus en plus
rarement.

---

## Partie 2 — Construction des N-grammes

Un **N-gramme** est une séquence de N tokens consécutifs. Pour la phrase
`<s> le chat mange du poisson </s>` (7 tokens) :

| Ordre | Nom | Nombre | Exemples |
|---|---|---|---|
| N = 1 | unigramme | 7 | `<s>`, `le`, `chat`, … |
| N = 2 | bigramme | 6 | `(<s>, le)`, `(le, chat)`, … |
| N = 3 | trigramme | 5 | `(<s>, le, chat)`, `(le, chat, mange)`, … |

Pour une phrase de *L* tokens, on obtient **L − N + 1** N-grammes d'ordre N.

Ces N-grammes sont **construits phrase par phrase**, jamais sur le corpus aplati :
un bigramme `(</s>, <s>)` à cheval sur deux phrases n'aurait aucun sens linguistique.

In [14]:
# Construction des trois niveaux de N-grammes
unigrammes  = construire_unigrammes(corpus)
bigrammes   = construire_bigrammes(corpus)
trigrammes  = construire_trigrammes(corpus)

afficher_ngrammes(unigrammes, "UNIGRAMMES")

UNIGRAMMES — 15 distincts, 45 occurrences

N-gramme  freq
--------------
le           9
</s>         6
<s>          6
chat         3
chien        3
aime         2
dans         2
jardin       2
joue         2
la           2
mange        2
poisson      2
viande       2
de           1
du           1


In [15]:
# --- Question 1 : tous les bigrammes et leurs fréquences ---
afficher_ngrammes(bigrammes, "BIGRAMMES")

BIGRAMMES — 23 distincts, 39 occurrences

N-gramme       freq
-------------------
<s> le            6
le chat           3
le chien          3
dans le           2
jardin </s>       2
joue dans         2
la viande         2
le jardin         2
poisson </s>      2
viande </s>       2
aime la           1
aime le           1
chat aime         1
chat joue         1
chat mange        1
chien aime        1
chien joue        1
chien mange       1
de la             1
du poisson        1
le poisson        1
mange de          1
mange du          1


In [16]:
# --- Question 2 : tous les trigrammes et leurs fréquences ---
afficher_ngrammes(trigrammes, "TRIGRAMMES")

TRIGRAMMES — 25 distincts, 33 occurrences

N-gramme           freq
-----------------------
<s> le chat           3
<s> le chien          3
dans le jardin        2
joue dans le          2
la viande </s>        2
le jardin </s>        2
aime la viande        1
aime le poisson       1
chat aime le          1
chat joue dans        1
chat mange du         1
chien aime la         1
chien joue dans       1
chien mange de        1
de la viande          1
du poisson </s>       1
le chat aime          1
le chat joue          1
le chat mange         1
le chien aime         1
le chien joue         1
le chien mange        1
le poisson </s>       1
mange de la           1
mange du poisson      1


In [17]:
# --- Questions 3 et 4 : N-grammes les plus fréquents ---
for nom, compteur in [("Bigramme", bigrammes), ("Trigramme", trigrammes)]:
    freq_max = max(compteur.values())
    gagnants = [ng for ng, f in compteur.items() if f == freq_max]
    print(f"{nom} le plus fréquent (freq = {freq_max}) :")
    for ng in sorted(gagnants):
        print(f"    ({', '.join(ng)})")
    print()

# Vérification arithmétique : nb de N-grammes = somme(L_i - N + 1)
print("Vérification des totaux")
for n, compteur in [(1, unigrammes), (2, bigrammes), (3, trigrammes)]:
    attendu = sum(len(p) - n + 1 for p in corpus)
    print(f"  N={n} : {sum(compteur.values())} occurrences "
          f"(attendu {attendu}) — {len(compteur)} distincts")

Bigramme le plus fréquent (freq = 6) :
    (<s>, le)

Trigramme le plus fréquent (freq = 3) :
    (<s>, le, chat)
    (<s>, le, chien)

Vérification des totaux
  N=1 : 45 occurrences (attendu 45) — 15 distincts
  N=2 : 39 occurrences (attendu 39) — 23 distincts
  N=3 : 33 occurrences (attendu 33) — 25 distincts


### Réponses — Partie 2

**Q1 et Q2.** Voir les tableaux ci-dessus : 23 bigrammes distincts pour 39 occurrences,
25 trigrammes distincts pour 33 occurrences.

**Q3. Bigramme le plus fréquent : `(<s>, le)`, fréquence 6.**

Il apparaît dans les 6 phrases : *toutes* commencent par `le`. Le modèle en déduira
`P(le | <s>) = 1` — le seul mot capable d'ouvrir une phrase. C'est vrai dans ce corpus,
faux en français : le corpus est trop petit pour que cette certitude soit légitime.

Si l'on écarte les marqueurs, les vainqueurs sont `(le, chat)` et `(le, chien)`,
fréquence 3 chacun.

**Q4. Trigramme le plus fréquent : `(<s>, le, chat)` et `(<s>, le, chien)`, fréquence 3 —
ex æquo.**

L'égalité n'est pas un hasard : le corpus a été construit symétriquement, trois phrases
sur le chat et trois sur le chien.

### Observation centrale : plus N augmente, plus les comptages s'effondrent

| Ordre | Occurrences | Distincts | Ratio occ./distinct |
|---|---|---|---|
| Unigrammes | 45 | 15 | 3.00 |
| Bigrammes | 39 | 23 | 1.70 |
| Trigrammes | 33 | 25 | 1.32 |

Deux mouvements opposés :

- **Les occurrences diminuent** (45 → 39 → 33), mécaniquement : chaque phrase de *L*
  tokens fournit L − N + 1 N-grammes, donc une de moins à chaque ordre.
- **Les distincts augmentent** (15 → 23 → 25), car il y a combinatoirement plus de
  séquences longues possibles que de mots isolés.

Résultat : le ratio chute de 3.00 à 1.32. **Sur 25 trigrammes, 19 n'apparaissent qu'une
seule fois.** Estimer une probabilité à partir d'une observation unique n'a aucune
robustesse statistique.

C'est le **problème de dispersion des données** (*data sparsity*), et il gouverne tout
le reste du TP : il explique pourquoi les probabilités nulles sont omniprésentes
(Partie 9), pourquoi le lissage est nécessaire (Partie 10), et pourquoi le trigramme
— pourtant plus informé — est plus fragile que le bigramme (Partie 11).

Théoriquement, avec V = 15, il existe 15² = 225 bigrammes possibles ; on n'en observe
que 23, soit **10 %**. Pour les trigrammes : 25 observés sur 3 375 possibles, soit
**0,7 %**.

---

## Partie 3 — Construction d'un modèle bigramme

### Le problème que résout l'hypothèse de Markov

La probabilité exacte d'une phrase se décompose par la **règle de la chaîne** :

$$P(w_1, \dots, w_n) = \prod_{i=1}^{n} P(w_i \mid w_1, \dots, w_{i-1})$$

Cette formule est exacte, mais inutilisable : estimer $P(w_i \mid w_1 \dots w_{i-1})$
exigerait d'avoir observé l'historique complet dans le corpus. Pour un historique de
10 mots et V = 10 000, cela ferait $10^{40}$ contextes distincts.

L'**hypothèse de Markov** tronque l'historique aux N−1 derniers mots. Pour un bigramme :

$$P(w_i \mid w_1, \dots, w_{i-1}) \approx P(w_i \mid w_{i-1})$$

C'est une approximation *fausse* linguistiquement — un mot dépend souvent de mots
lointains — mais elle rend le modèle estimable.

### Estimation par maximum de vraisemblance

$$P(w_i \mid w_{i-1}) = \frac{C(w_{i-1}, w_i)}{C(w_{i-1})}$$

On divise le nombre de fois où le bigramme a été vu par le nombre de fois où le mot
de contexte a été vu. C'est la fréquence relative observée : l'estimateur qui maximise
la vraisemblance du corpus d'entraînement.

**Point de vigilance sur le dénominateur.** On divise par $C(w_{i-1})$, le comptage
**unigramme**, et non par le nombre de bigrammes commençant par $w_{i-1}$. Les deux
coïncident presque, mais pas exactement : `</s>` termine une phrase et n'ouvre jamais
de bigramme. Cette petite différence garantit que les probabilités somment à 1 pour
tout contexte autre que `</s>`.

In [18]:
# Entraînement du modèle bigramme
modele = ModeleNgramme(corpus)

print(f"Modèle entraîné : {len(modele.corpus)} phrases, "
      f"N = {modele.N} tokens, V = {modele.V} types")

Modèle entraîné : 6 phrases, N = 45 tokens, V = 15 types


In [19]:
# --- Les six probabilités demandées par le TP ---
paires = [
    ("le", "chat"), ("le", "chien"),
    ("chat", "mange"), ("chat", "aime"),
    ("du", "poisson"), ("la", "viande"),
]

for precedent, mot in paires:
    print(modele.detail_probabilite(precedent, mot))

P(chat | le) = C(le, chat) / C(le) = 3/9 = 0.3333
P(chien | le) = C(le, chien) / C(le) = 3/9 = 0.3333
P(mange | chat) = C(chat, mange) / C(chat) = 1/3 = 0.3333
P(aime | chat) = C(chat, aime) / C(chat) = 1/3 = 0.3333
P(poisson | du) = C(du, poisson) / C(du) = 1/1 = 1.0000
P(viande | la) = C(la, viande) / C(la) = 2/2 = 1.0000


In [20]:
# --- Distributions de probabilité pour quelques contextes ---
for contexte in ["<s>", "le", "chat", "chien", "mange", "joue"]:
    distribution = modele.successeurs(contexte)
    total = sum(distribution.values())
    print(f"Après « {contexte} »  (C = {modele.compte_unigramme(contexte)}) :")
    for mot, p in distribution.items():
        barre = "█" * int(p * 30)
        print(f"    P({mot:8s}| {contexte:5s}) = {p:.4f}  {barre}")
    print(f"    → somme = {total:.4f}\n")

Après « <s> »  (C = 6) :
    P(le      | <s>  ) = 1.0000  ██████████████████████████████
    → somme = 1.0000

Après « le »  (C = 9) :
    P(chat    | le   ) = 0.3333  ██████████
    P(chien   | le   ) = 0.3333  ██████████
    P(jardin  | le   ) = 0.2222  ██████
    P(poisson | le   ) = 0.1111  ███
    → somme = 1.0000

Après « chat »  (C = 3) :
    P(aime    | chat ) = 0.3333  ██████████
    P(joue    | chat ) = 0.3333  ██████████
    P(mange   | chat ) = 0.3333  ██████████
    → somme = 1.0000

Après « chien »  (C = 3) :
    P(aime    | chien) = 0.3333  ██████████
    P(joue    | chien) = 0.3333  ██████████
    P(mange   | chien) = 0.3333  ██████████
    → somme = 1.0000

Après « mange »  (C = 2) :
    P(de      | mange) = 0.5000  ███████████████
    P(du      | mange) = 0.5000  ███████████████
    → somme = 1.0000

Après « joue »  (C = 2) :
    P(dans    | joue ) = 1.0000  ██████████████████████████████
    → somme = 1.0000



In [21]:
# --- Quelques probabilités nulles, pour la Question 1 ---
print("Bigrammes jamais observés :\n")
for precedent, mot in [("chat", "pain"), ("chat", "le"),
                       ("le", "mange"), ("viande", "poisson")]:
    print("   ", modele.detail_probabilite(precedent, mot))

Bigrammes jamais observés :

    P(pain | chat) = C(chat, pain) / C(chat) = 0/3 = 0.0000
    P(le | chat) = C(chat, le) / C(chat) = 0/3 = 0.0000
    P(mange | le) = C(le, mange) / C(le) = 0/9 = 0.0000
    P(poisson | viande) = C(viande, poisson) / C(viande) = 0/2 = 0.0000


### Réponses — Partie 3

**Q1. Pourquoi certaines probabilités sont-elles nulles ?**

Parce que le numérateur $C(w_{i-1}, w_i)$ vaut 0 : le bigramme **n'a jamais été observé
dans le corpus**. Exemple : $P(\text{pain} \mid \text{chat}) = 0/3 = 0$.

Il faut bien distinguer deux causes très différentes, que le modèle confond :

- **L'impossibilité linguistique** — `(viande, poisson)` est effectivement mal formé
  en français.
- **L'absence d'observation** — `(chat, pain)` est parfaitement correct en français,
  mais notre corpus de 6 phrases ne parle pas de pain.

Le modèle ne fait aucune différence entre les deux : il attribue 0 dans les deux cas.
C'est une **erreur d'estimation**, pas une vérité linguistique. Un corpus de 45 tokens
ne peut pas prétendre couvrir le français.

Chiffré : avec V = 15, il existe 225 bigrammes possibles ; 23 sont observés.
**202 bigrammes, soit 90 %, ont une probabilité nulle.**

**Q2. Que signifie une probabilité élevée pour un bigramme ?**

Que le second mot suit très régulièrement le premier **dans ce corpus**. Trois nuances :

- $P(\text{poisson} \mid \text{du}) = 1/1 = 1$ : le maximum absolu, mais fondé sur
  **une seule** observation. Statistiquement, cela ne vaut rien — c'est un artefact de
  la petite taille du corpus, pas une régularité du français.
- $P(\text{viande} \mid \text{la}) = 2/2 = 1$ : même certitude, sur 2 observations.
  À peine mieux.
- $P(\text{chat} \mid \text{le}) = 3/9 = 0.33$ : plus faible, mais reposant sur
  9 observations du contexte — **l'estimation la plus fiable des trois**.

Leçon : une probabilité élevée n'est informative que si le **dénominateur** est grand.
La valeur seule ne dit rien ; il faut regarder sur combien d'observations elle repose.

**Q3. Que signifie une probabilité nulle ?**

Littéralement : « ce bigramme n'a jamais été vu ». Le modèle le traite comme
**impossible**, ce qui est bien plus fort.

La conséquence est grave, car les probabilités de phrase sont des **produits** :

$$P(S) = \prod_i P(w_i \mid w_{i-1})$$

Un seul facteur nul annule tout le produit. Une phrase de 20 mots parfaitement
grammaticale reçoit $P(S) = 0$ à cause d'un seul bigramme jamais rencontré. Le modèle
devient alors incapable de comparer deux phrases — elles sont toutes deux à zéro.

C'est le **problème des comptes nuls**, traité en Partie 9 et résolu par le lissage de
Laplace en Partie 10.

### Ce que révèlent les distributions

Deux formes très différentes cohabitent :

- **Distributions plates** : après `chat`, les trois successeurs `aime`, `joue`, `mange`
  sont à 1/3 chacun. Le contexte n'apporte aucune information discriminante — le modèle
  est indécis.
- **Distributions dégénérées** : après `<s>`, `le` est à 1.0 ; après `du` ou `dans`,
  un seul successeur. Le modèle est *certain*, mais cette certitude vient de la pauvreté
  du corpus, pas d'une régularité de la langue.

Un vrai corpus produit des distributions intermédiaires : un successeur dominant, une
longue traîne de successeurs rares. Ici on n'a que les deux extrêmes.

---

## Partie 4 — Tâche NLP 1 : prédiction du mot suivant

C'est l'application la plus visible d'un modèle de langage : le clavier prédictif
d'un téléphone. L'utilisateur tape un contexte, le modèle propose les mots les plus
probables.

Le principe :

$$\hat{w} = \arg\max_{w \in V} P(w \mid w_{i-1})$$

On parcourt les successeurs observés du dernier mot et on retient celui de plus forte
probabilité.

**Conséquence directe de l'hypothèse de Markov :** dans un modèle bigramme, seul le
**dernier mot** du contexte compte. Prédire après « le chat » et après « chat » donne
strictement le même résultat — le mot `le` est ignoré.

In [22]:
# --- Tests demandés par le TP ---
for contexte in ["le chat", "le chien", "le", "chat"]:
    modele.afficher_prediction(contexte)

Contexte : « le chat »   -> mot conditionnant : « chat »  (C = 3)
    P(aime    | chat  ) = 0.3333  ##########
    P(joue    | chat  ) = 0.3333  ##########
    P(mange   | chat  ) = 0.3333  ##########
    => ÉGALITÉ entre aime, joue, mange -> choix alphabétique : « aime »

Contexte : « le chien »   -> mot conditionnant : « chien »  (C = 3)
    P(aime    | chien ) = 0.3333  ##########
    P(joue    | chien ) = 0.3333  ##########
    P(mange   | chien ) = 0.3333  ##########
    => ÉGALITÉ entre aime, joue, mange -> choix alphabétique : « aime »

Contexte : « le »   -> mot conditionnant : « le »  (C = 9)
    P(chat    | le    ) = 0.3333  ##########
    P(chien   | le    ) = 0.3333  ##########
    P(jardin  | le    ) = 0.2222  ######
    P(poisson | le    ) = 0.1111  ###
    => ÉGALITÉ entre chat, chien -> choix alphabétique : « chat »

Contexte : « chat »   -> mot conditionnant : « chat »  (C = 3)
    P(aime    | chat  ) = 0.3333  ##########
    P(joue    | chat  ) = 0.3333  ##########
  

In [23]:
# --- Deux cas limites intéressants ---
for contexte in ["<s>", "jardin", "poisson", "pain"]:
    modele.afficher_prediction(contexte)

Contexte : « <s> »   -> mot conditionnant : « <s> »  (C = 6)
    P(le      | <s>   ) = 1.0000  ##############################
    => mot le plus probable : « le »

Contexte : « jardin »   -> mot conditionnant : « jardin »  (C = 2)
    P(</s>    | jardin) = 1.0000  ##############################
    => mot le plus probable : « </s> »

Contexte : « poisson »   -> mot conditionnant : « poisson »  (C = 2)
    P(</s>    | poisson) = 1.0000  ##############################
    => mot le plus probable : « </s> »

Contexte : « pain »   -> mot conditionnant : « pain »  (C = 0)
    aucun successeur observé : le modèle ne peut rien prédire



In [24]:
# --- Question : pourquoi P(chat | le) ≠ P(le | chat) ? ---
print("Les deux directions reposent sur des comptages différents :\n")
print(f"  C(le, chat) = {modele.compte_bigramme('le', 'chat')}   "
      f"C(le)   = {modele.compte_unigramme('le')}")
print(f"  C(chat, le) = {modele.compte_bigramme('chat', 'le')}   "
      f"C(chat) = {modele.compte_unigramme('chat')}\n")
print(modele.detail_probabilite("le", "chat"))
print(modele.detail_probabilite("chat", "le"))

Les deux directions reposent sur des comptages différents :

  C(le, chat) = 3   C(le)   = 9
  C(chat, le) = 0   C(chat) = 3

P(chat | le) = C(le, chat) / C(le) = 3/9 = 0.3333
P(le | chat) = C(chat, le) / C(chat) = 0/3 = 0.0000


### Réponses — Partie 4

**Question : différence entre $P(\text{chat} \mid \text{le})$ et $P(\text{le} \mid \text{chat})$.
Pourquoi ne sont-elles généralement pas égales ?**

Ce sont deux quantités qui ne posent pas la même question :

- $P(\text{chat} \mid \text{le})$ : « sachant que je viens de lire `le`, quelle chance
  que le mot suivant soit `chat` ? » → $C(\text{le}, \text{chat}) / C(\text{le}) = 3/9 = 0.333$
- $P(\text{le} \mid \text{chat})$ : « sachant que je viens de lire `chat`, quelle chance
  que le mot suivant soit `le` ? » → $C(\text{chat}, \text{le}) / C(\text{chat}) = 0/3 = 0$

**Trois raisons de l'asymétrie :**

1. **Les dénominateurs diffèrent.** $C(\text{le}) = 9$ contre $C(\text{chat}) = 3$.
   Même à numérateur identique, les résultats divergeraient.

2. **Les numérateurs diffèrent, car un bigramme est ordonné.** $(le, chat)$ et
   $(chat, le)$ sont deux N-grammes **distincts** : le premier apparaît 3 fois, le
   second jamais. C'est précisément ce qui permet aux N-grammes de capturer l'ordre
   des mots (voir Partie 7).

3. **Linguistiquement, le français est asymétrique.** Un déterminant précède son nom ;
   `le chat` est bien formé, `chat le` ne l'est pas.

Formellement, l'égalité n'aurait lieu que si $C(w_1) = C(w_2)$, par la règle de Bayes :

$$P(A \mid B) = \frac{P(B \mid A) \, P(A)}{P(B)}$$

Les deux ne coïncident que lorsque $P(A) = P(B)$ — cas exceptionnel.

**Confondre les deux est l'erreur la plus fréquente sur ce chapitre.** Un modèle de
langage lit de gauche à droite : c'est toujours le contexte *passé* qui conditionne
le mot *à venir*, jamais l'inverse.

### Deux observations sur les prédictions

**Les égalités massives.** Après `chat`, les trois candidats sont à exactement 1/3.
Le modèle est incapable de trancher, et la « prédiction » n'est qu'une convention de
départage alphabétique. Avec un corpus réaliste, les fréquences se différencient
naturellement et ce cas devient rare.

**Le mot hors vocabulaire.** Après `pain` — absent du corpus — le modèle ne retourne
**rien du tout** : $C(\text{pain}) = 0$, donc aucun successeur, et le calcul serait une
division par zéro. Un modèle de production traite ce cas avec un token spécial `<UNK>`
ou un mécanisme de repli (*backoff*) vers l'unigramme. Ici, le modèle est simplement
muet.